# 3.1 Training Pipeline

### 3.1.1 Configuration

In [1]:
import json
import time
from pathlib import Path
from typing import Any

import cv2
import numpy as np
from sklearn.metrics import accuracy_score

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = Path.cwd().parent

data_root = project_root / "data"
models_root = project_root / "models"
models_root.mkdir(parents=True, exist_ok=True)


def require_dir(root: Path, folder_name: str, guidance: str) -> Path:
    path = root / folder_name
    if not path.exists() or not path.is_dir():
        raise FileNotFoundError(
            f"Required folder not found: {path}. {guidance}"
        )
    return path


def create_lbph_recognizer() -> Any:
    face_module = getattr(cv2, "face", None)
    factory = getattr(face_module, "LBPHFaceRecognizer_create", None)
    if factory is None:
        raise RuntimeError(
            "LBPH is unavailable in this kernel. Run Cell 2 and restart kernel if needed."
        )
    return factory()


# Inputs from 2_augmentation_pipeline (strict, no fallback)
AUGMENTED_TRAIN_DIR = require_dir(
    data_root,
    "7_augmented",
    "Run processing/2_augmentation_pipeline.ipynb first to generate required folders.",
)
BASELINE_TRAIN_DIR = require_dir(
    data_root,
    "7_final_processed",
    "Run processing/2_augmentation_pipeline.ipynb first to generate required folders.",
)
TEST_PROCESSED_DIR = require_dir(
    data_root,
    "7_test_processed",
    "Run processing/2_augmentation_pipeline.ipynb first to generate required folders.",
)

# Identity model paths
LBPH_BASELINE_PATH = models_root / "lbph_model.yml"
LBPH_AUG_PATH = models_root / "lbph_with_aug.yml"
LBPH_FINAL_PATH = models_root / "lbph_final.yml"
LABEL_MAP_PATH = models_root / "label_map.txt"

# Runtime bundle path
RUNTIME_CONFIG_PATH = models_root / "realtime_model_config.json"

OUTPUT_SIZE = (128, 128)
LBPH_CONFIDENCE_THRESHOLD = 160.0
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Identity training pipeline configured")
print(f"  Augmented train : {AUGMENTED_TRAIN_DIR}")
print(f"  Baseline train  : {BASELINE_TRAIN_DIR}")
print(f"  Test processed  : {TEST_PROCESSED_DIR}")
print(f"  Models output   : {models_root}")

Identity training pipeline configured
  Augmented train : c:\Users\harry\Documents\school\ip\AttSystem\data\7_augmented
  Baseline train  : c:\Users\harry\Documents\school\ip\AttSystem\data\7_final_processed
  Test processed  : c:\Users\harry\Documents\school\ip\AttSystem\data\7_test_processed
  Models output   : c:\Users\harry\Documents\school\ip\AttSystem\models


### 3.1.2 Helper Functions

In [2]:
def read_gray(path: Path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    if img.shape != OUTPUT_SIZE:
        img = cv2.resize(img, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)
    return img


def load_identity_dataset(train_folder: Path, test_folder: Path):
    if not train_folder.exists():
        raise FileNotFoundError(f"Training folder not found: {train_folder}")

    person_dirs = sorted([p for p in train_folder.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError(f"No person folders found in: {train_folder}")

    label_names = []
    label_map = {}
    X_train, y_train = [], []
    X_test, y_test = [], []

    for label_id, person_dir in enumerate(person_dirs):
        person_name = person_dir.name
        label_names.append(person_name)
        label_map[person_name] = label_id

        train_files = sorted([
            f for f in person_dir.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS
        ])

        loaded_train = 0
        for file_path in train_files:
            img = read_gray(file_path)
            if img is None:
                continue
            X_train.append(img)
            y_train.append(label_id)
            loaded_train += 1

        test_person_dir = test_folder / person_name
        loaded_test = 0
        if test_person_dir.exists() and test_person_dir.is_dir():
            test_files = sorted([
                f for f in test_person_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTS
            ])
            for file_path in test_files:
                img = read_gray(file_path)
                if img is None:
                    continue
                X_test.append(img)
                y_test.append(label_id)
                loaded_test += 1

        print(f"{person_name}: train={loaded_train}, test={loaded_test}")

    return {
        "X_train": np.array(X_train),
        "y_train": np.array(y_train, dtype=np.int32),
        "X_test": np.array(X_test),
        "y_test": np.array(y_test, dtype=np.int32),
        "label_names": label_names,
        "label_map": label_map,
    }


def train_lbph(dataset, save_path: Path, radius=1, neighbors=8, grid_x=8, grid_y=8):
    if len(dataset["X_train"]) == 0:
        raise RuntimeError("Empty training set.")

    model = create_lbph_recognizer()
    model.setRadius(radius)
    model.setNeighbors(neighbors)
    model.setGridX(grid_x)
    model.setGridY(grid_y)

    t0 = time.time()
    model.train(list(dataset["X_train"]), dataset["y_train"])
    elapsed = time.time() - t0

    save_path.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(save_path))

    metrics = {
        "train_time": elapsed,
        "accuracy": None,
        "mean_confidence": None,
    }

    if len(dataset["X_test"]) > 0:
        preds = []
        confs = []
        for img in dataset["X_test"]:
            p, c = model.predict(img)
            preds.append(p)
            confs.append(c)
        preds = np.array(preds)
        confs = np.array(confs, dtype=np.float32)
        metrics["accuracy"] = float(accuracy_score(dataset["y_test"], preds) * 100.0)
        metrics["mean_confidence"] = float(np.mean(confs))

    return model, metrics


def save_label_map(label_names, save_path: Path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        for idx, name in enumerate(label_names):
            f.write(f"{idx},{name}\n")

### 3.1.3 LBPH Training

In [3]:
import gc

baseline_data = load_identity_dataset(BASELINE_TRAIN_DIR, TEST_PROCESSED_DIR)
augmented_data = load_identity_dataset(AUGMENTED_TRAIN_DIR, TEST_PROCESSED_DIR)

def evaluate_lbph(model, dataset):
    if len(dataset["X_test"]) == 0:
        return None, None
    preds, confs = [], []
    for img in dataset["X_test"]:
        p, c = model.predict(img)
        preds.append(p)
        confs.append(c)
    preds = np.array(preds)
    confs = np.array(confs, dtype=np.float32)
    acc = float(accuracy_score(dataset["y_test"], preds) * 100.0)
    mean_conf = float(np.mean(confs))
    return acc, mean_conf

# Baseline model
baseline_model = create_lbph_recognizer()
baseline_model.setRadius(1)
baseline_model.setNeighbors(8)
baseline_model.setGridX(8)
baseline_model.setGridY(8)
baseline_model.train(list(baseline_data["X_train"]), baseline_data["y_train"])
LBPH_BASELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
baseline_model.save(str(LBPH_BASELINE_PATH))
baseline_acc, baseline_conf = evaluate_lbph(baseline_model, baseline_data)
baseline_metrics = {"accuracy": baseline_acc, "mean_confidence": baseline_conf}

# Release baseline objects before heavier training
del baseline_model
del baseline_data
gc.collect()

# Augmented model
aug_model = create_lbph_recognizer()
aug_model.setRadius(1)
aug_model.setNeighbors(8)
aug_model.setGridX(8)
aug_model.setGridY(8)
aug_model.train(list(augmented_data["X_train"]), augmented_data["y_train"])
LBPH_AUG_PATH.parent.mkdir(parents=True, exist_ok=True)
aug_model.save(str(LBPH_AUG_PATH))
aug_acc, aug_conf = evaluate_lbph(aug_model, augmented_data)
aug_metrics = {"accuracy": aug_acc, "mean_confidence": aug_conf}

# Final tuned model uses a safer LBPH configuration.
# A 16-neighbor LBPH model can explode histogram size and appear to hang on larger sets.
print("Training final tuned model (radius=2, neighbors=8)...")
final_model = create_lbph_recognizer()
final_model.setRadius(2)
final_model.setNeighbors(8)
final_model.setGridX(8)
final_model.setGridY(8)

try:
    gc.collect()
    final_model.train(list(augmented_data["X_train"]), augmented_data["y_train"])
    final_acc, final_conf = evaluate_lbph(final_model, augmented_data)
    final_metrics = {"accuracy": final_acc, "mean_confidence": final_conf}
except (cv2.error, MemoryError) as err:
    print(f"Final LBPH training failed ({err}).")
    print("Falling back to the augmented model for runtime final model.")
    final_model = aug_model
    final_acc, final_conf = aug_acc, aug_conf
    final_metrics = {
        "accuracy": final_acc,
        "mean_confidence": final_conf,
        "fallback": "lbph_with_aug.yml",
    }

LBPH_FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)
final_model.save(str(LBPH_FINAL_PATH))

save_label_map(augmented_data["label_names"], LABEL_MAP_PATH)

print("\nIdentity models saved")
print(f"  baseline : {LBPH_BASELINE_PATH} | acc={baseline_metrics['accuracy']}")
print(f"  with_aug : {LBPH_AUG_PATH} | acc={aug_metrics['accuracy']}")
print(f"  final    : {LBPH_FINAL_PATH} | acc={final_metrics['accuracy']}")
if "fallback" in final_metrics:
    print(f"  final mode: fallback -> {final_metrics['fallback']}")
print(f"  label map: {LABEL_MAP_PATH}")

benjamin: train=13, test=4
chern_tak: train=22, test=6
chillien: train=6, test=2
daniel: train=15, test=4
dylan: train=12, test=1
han_soon: train=15, test=4
harry: train=7, test=2
isaac: train=6, test=2
jing_ang: train=20, test=5
jun_wei: train=2, test=0
kang_kai: train=17, test=6
marion: train=22, test=6
ms_nurul: train=8, test=2
qi_xuan: train=24, test=6
shuang_quan: train=22, test=9
wee_xuan: train=19, test=5
xiang_yue: train=6, test=3
xu_sheng: train=23, test=6
yoke_hong: train=13, test=3
yong_kang: train=19, test=5
zi_herng: train=8, test=2
benjamin: train=100, test=4
chern_tak: train=100, test=6
chillien: train=100, test=2
daniel: train=100, test=4
dylan: train=100, test=1
han_soon: train=100, test=4
harry: train=100, test=2
isaac: train=100, test=2
jing_ang: train=100, test=5
jun_wei: train=100, test=0
kang_kai: train=100, test=6
marion: train=100, test=6
ms_nurul: train=100, test=2
qi_xuan: train=100, test=6
shuang_quan: train=100, test=9
wee_xuan: train=100, test=5
xiang_yue: 

In [4]:
runtime_config = {
    "identity_model": str(LBPH_FINAL_PATH),
    "identity_label_map": str(LABEL_MAP_PATH),
    "identity_confidence_threshold": LBPH_CONFIDENCE_THRESHOLD,
    "input_size": list(OUTPUT_SIZE),
    "notes": {
        "identity": "LBPH expects aligned face ROI in grayscale.",
        "training_flow": "lbph_model.yml and lbph_with_aug.yml are comparison checkpoints; lbph_final.yml is the model used at runtime.",
    },
}

with open(RUNTIME_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

print(f"Runtime config saved: {RUNTIME_CONFIG_PATH}")
print(json.dumps(runtime_config, indent=2))


def load_label_map(path: Path):
    mapping = {}
    if not path.exists():
        return mapping
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            idx, name = line.split(",", 1)
            mapping[int(idx)] = name
    return mapping


def quick_predict_face(face_gray: np.ndarray):
    face_gray = cv2.resize(face_gray, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)

    recognizer = create_lbph_recognizer()
    recognizer.read(str(LBPH_FINAL_PATH))

    label_map = load_label_map(LABEL_MAP_PATH)
    pred_id, confidence = recognizer.predict(face_gray)
    person_name = label_map.get(pred_id, "unknown")
    if confidence > LBPH_CONFIDENCE_THRESHOLD:
        person_name = "unknown"

    return {
        "name": person_name,
        "identity_confidence": float(confidence),
    }

# Smoke test on one sample from test_processed (if available)
smoke_done = False
if TEST_PROCESSED_DIR.exists():
    for class_dir in sorted([p for p in TEST_PROCESSED_DIR.iterdir() if p.is_dir()]):
        sample_files = sorted([f for f in class_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not sample_files:
            continue
        sample = cv2.imread(str(sample_files[0]), cv2.IMREAD_GRAYSCALE)
        if sample is None:
            continue
        print("Smoke test result:")
        print(quick_predict_face(sample))
        smoke_done = True
        break

if not smoke_done:
    print("No sample available for smoke test in resolved test folder.")

Runtime config saved: c:\Users\harry\Documents\school\ip\AttSystem\models\realtime_model_config.json
{
  "identity_model": "c:\\Users\\harry\\Documents\\school\\ip\\AttSystem\\models\\lbph_final.yml",
  "identity_label_map": "c:\\Users\\harry\\Documents\\school\\ip\\AttSystem\\models\\label_map.txt",
  "identity_confidence_threshold": 160.0,
  "input_size": [
    128,
    128
  ],
  "notes": {
    "identity": "LBPH expects aligned face ROI in grayscale.",
    "training_flow": "lbph_model.yml and lbph_with_aug.yml are comparison checkpoints; lbph_final.yml is the model used at runtime."
  }
}
Smoke test result:
{'name': 'benjamin', 'identity_confidence': 82.46323750223976}
